RAG Architecture

user query(input question) -> retriever(finds relevant docs) -> (The retrieved documents become context) context + query(combined prompt) -> LLM (generates answer) -> response(grounded in docs)

explains how a user's question is converted into an answer using external knowledge (documents) instead of relying only on the LLM's internal training.

.md files are Markdown files.

Markdown is a lightweight markup language used to write formatted text using simple plain-text syntax.

In [ ]:
#enumerate() starts counting at 0, but people usually want numbering starting from 1.
#Python loads these classes and functions into memory.
from langchain_core import documents
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv
import tempfile

"""Install with: pip install langchain langchain-openai
Create your first chain in under 10 lines of code."""

embeddings_model=OpenAIEmbeddings(model="text-embedding-3-small")

def create_kb():
    """
    Its purpose is to prepare the knowledge base for storage.
    Create a vector store from knowledge base.

    Take the knowledge base text.
    Split it into pieces.
    Convert those pieces into embeddings.
    Store them in a vector database (later).

    If your knowledge base is very large (for example, 1000 pages), sending the whole document to an LLM every time would:

    exceed token limits,
    increase latency,
    cost more.

    Instead:

    Split into chunks.
    Convert each chunk into an embedding.
    Store embeddings in a vector database.
    When a user asks a question, retrieve only the most relevant chunks.
    Send only those chunks to the LLM.

    This is the foundation of a typical RAG pipeline:

    """
    splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
    )

    doc = Document(
    page_content=KNOWLEDGE_BASE,
    metadata={"source": "langchain_knowledge_base.md"} #Later, if the retriever returns an answer, you can know where it came from.
    )

    #The result, chunks, is a list of smaller Document objects, not plain strings.
    chunks = splitter.split_documents([doc]) #split_documents() expects a list of Document objects, even if you only have one.

    #create a vectorstore 
    vectorstore= Chroma.from_documents(
        documents=chunks,
        embedding=embeddings_model,
        persist_directory=tempfile.mkdtemp(),
    )

    return vectorstore

def demo_basic_rag():

    vectorstore=create_kb()

    retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":2})

    llm=init_chat_model(
        model="gpt-4o-mini",
        temperature=0.2,
    )

    #RAG Prompt Template

    #Grounding to prevent hallucinations
    prompt=ChatPromptTemplate.from_template(
        """ Answer the question based on only the following context:
        {context}
        Question: {question}
        Make sure to answer in a concise manner, and if you don't know the answer then just say "I don't have that information in my knowledge base" """
    )

    #Format retrieved documents
    def format_docs(docs):
        return "\n\n".join([doc.page_content for doc in docs])

    #RAG CHAIN
    #the dictionary stores runnables (things that can process input).
    rag_chain=(
        {
          "context": retriever | format_docs,
           "question": RunnablePassthrough()
        }

        | prompt | llm | StrOutputParser() #Because of StrOutputParser(), The AIMessage object becomes a plain Python string.

    )

    # Test
    questions = [
        "What is LangChain?",
        "Who created LangChain?",
        "What is LangGraph used for?",
        "How do I deploy LangChain to AWS?" #Not in the knowledge base
    ]

    print("Basic RAG Demo:\n")

    for q in questions:
        answer = rag_chain.invoke(q)
        print(f"Q: {q}")
        print(f"A: {answer}\n")


def format_docs_with_sources(docs):
    formatted = []

    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown")
        formatted.append(f"[{i + 1}] {source}:\n{doc.page_content}")

    return "\n\n".join(formatted)

    
if __name__=="__main__":
    demo_basic_rag()


RAG with Fallback (graceful handling of unknown questions)

RAG with Structured Output

In [ ]:
from langchain_core.outputs import chat_result
def demo_structured_rag():

    """RAG with structured output."""

    vectorstore = create_kb()

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    class RAGResponse (BaseModel):
        """Structured RAG response."""
        answer: str = Field(description="The answer to the question")
        confidence: str=Field(description="high, medium, or low")
        sources_used: List [str] = Field(description="List of sources referenced")
        follow_up: str = Field(description="Suggested follow-up question")
    
    structured_llm=llm.with_structured_output(RAGResponse)

    prompt=ChatPromptTemplate.from_template("""
    Based on the context below asnwer the question:
    Context:{context}
    Question:{question}
    Provide a structured response.
    """)

    def format_docs(docs):
        return "\n\n".join(f"[{doc.metadata.get('source','unknown')}]: {doc.page_content}" for doc in docs)
    
    rag_chain= ({"context":retriever |format_docs,
                "question":RunnablePassthrough()}
                | prompt | structured_llm )

    print("Structured RAG Demo:\n")
    result = rag_chain.invoke("What is LangGraph?")
    print(f"Answer: {result.answer}")
    print(f"Confidence: {result.confidence}")
    print(f"Sources: {result.sources_used}")
    print(f"Follow-up: {result.follow_up}")


if __name__ == "__main__":
    # demo_basic_rag()
    # demo_rag_with_sources()
    # demo_rag_with_fallback()
    demo_structured_rag()   